In [1]:
# Cell 1: Setup and Imports
import os
import sys
import yaml
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings('ignore')

# Safely resolve project root
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.models.ncf_model import NeuralCFRecommender, NCFDataset
from mlops.train import prepare_sparse_matrices
from mlops.evaluate import evaluate_model_at_k

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Ready to train on: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

[INFO] Ready to train on: NVIDIA GeForce RTX 4070 Laptop GPU


In [2]:
# Cell 2: Load and Prepare Data
CONFIG_PATH = os.path.join(PROJECT_ROOT, 'configs', 'data_config.yaml')
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    data_config = yaml.safe_load(f)

PROCESSED_DIR = os.path.join(PROJECT_ROOT, data_config['paths']['processed_data'])

print("[INFO] Loading datasets...")
train_df = pd.read_parquet(os.path.join(PROCESSED_DIR, data_config['paths']['train_file']))
val_df = pd.read_parquet(os.path.join(PROCESSED_DIR, data_config['paths']['val_file']))

# Filter 2017+ to match baseline comparisons
train_df = train_df[train_df['timestamp'] >= '2017-01-01']

print("[INFO] Preparing sparse matrices...")
train_matrix, val_matrix, eval_users = prepare_sparse_matrices(train_df, val_df)

num_users, num_items = train_matrix.shape
print(f"[INFO] Matrix Shape: {num_users} Users x {num_items} Items")

[INFO] Loading datasets...
[INFO] Preparing sparse matrices...
[INFO] Matrix Shape: 997856 Users x 9880 Items


In [3]:
# Cell 3: Initialize PyTorch Dataset and DataLoader
BATCH_SIZE = 2048
NUM_NEGATIVES = 4

print("[INFO] Initializing NCF Dataset...")
train_dataset = NCFDataset(train_matrix, num_items, num_negatives=NUM_NEGATIVES)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=0
)

print(f"[INFO] DataLoader ready with {len(train_loader)} batches.")

[INFO] Initializing NCF Dataset...
[INFO] Generating 4 negative samples per positive interaction...
[INFO] Dataset creation completed. Total samples: 10828970
[INFO] DataLoader ready with 5288 batches.


In [4]:
# Cell 4: Initialize and Train NCF Model
EMBEDDING_DIM = 32
EPOCHS = 5
LEARNING_RATE = 0.001

print("[INFO] Initializing NCF Model Wrapper...")
ncf_recommender = NeuralCFRecommender(num_users=num_users, num_items=num_items, emb_dim=EMBEDDING_DIM)
ncf_recommender.fit(train_matrix) 

# Extract the inner PyTorch network and set up optimization artifacts
model = ncf_recommender.model
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.BCELoss()

print("[INFO] Starting Deep Learning Training Loop on GPU...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
    
    for batch_users, batch_items, batch_labels in progress_bar:
        batch_users = batch_users.to(device).long()
        batch_items = batch_items.to(device).long()
        batch_labels = batch_labels.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass through the MLP layers
        predictions = model(batch_users, batch_items)
        
        # Calculate Binary Cross Entropy loss
        loss = criterion(predictions, batch_labels)
        
        # Backward pass and weight updates
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch}/{EPOCHS} Completed | Average Loss: {avg_loss:.4f}")

[INFO] Initializing NCF Model Wrapper...
[INFO] NCF model allocated on device: cuda
[INFO] Starting Deep Learning Training Loop on GPU...


Epoch 1/5:   0%|          | 0/5288 [00:00<?, ?it/s]

Epoch 1/5 Completed | Average Loss: 0.3990


Epoch 2/5:   0%|          | 0/5288 [00:00<?, ?it/s]

Epoch 2/5 Completed | Average Loss: 0.3809


...

Epoch 3/5 Completed | Average Loss: 0.3796


Epoch 4/5:   0%|          | 0/5288 [00:00<?, ?it/s]

Epoch 4/5 Completed | Average Loss: 0.3774


Epoch 5/5:   0%|          | 0/5288 [00:00<?, ?it/s]

Epoch 5/5 Completed | Average Loss: 0.3731


In [5]:
# Cell 5: Evaluate NCF Model Performance
print("\n[INFO] Evaluating Neural CF Model at K=10...")

# Evaluates the model using the unified project evaluation pipeline
ncf_metrics = evaluate_model_at_k(ncf_recommender, val_matrix, eval_users, k=10)

print("\n" + "="*40)
print("Neural Collaborative Filtering (NCF) Results")
print("="*40)
for metric, value in ncf_metrics.items():
    print(f"{metric}: {value:.4f}")


[INFO] Evaluating Neural CF Model at K=10...



Neural Collaborative Filtering (NCF) Results
HitRate_10: 0.0455
Precision_10: 0.0047
Adj_Precision_10: 0.0313
Recall_10: 0.0313
MRR_10: 0.0123
NDCG_10: 0.0152
